In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [2]:
load_dotenv()

False

In [3]:
model = ChatOpenAI(model = 'gpt-4o-mini')

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
class SentimentSchema(BaseModel):
    sentiment : Literal['positive','negative'] = Field(description = 'Sentence of the review')

In [ ]:
class DiagnosticSchema(BaseModel):
    issue_type: Literal['UX','Performance','Bug','Support','Other'] = Field(description='The category of review')
    tone : Literal['angry','frustated','disappointed','calm'] = Field(description="The emotion of the reviewer")
    urgency: Literal["low",'medium','high'] = Field(description='How urgent or critical the issue is')

In [ ]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model_2 = model.with_structured_output(DiagnosticSchema)

In [ ]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal['positive','negative']
    diagnosis:str
    response:str

In [ ]:
def find_sentiment(state:ReviewState):
    text = state['review']
    prompt = f"Please classify the sentiment of the following review as either 'positive' or 'negative':\n\n{text}"
    result = structured_model.invoke(prompt)
    return {'sentiment': result.sentiment}

def diagnose_sentiment(state:ReviewState):
    prompt = f"Diagnose this negative review {state['review']} and return issue_type,tone and urgency"
    result = structured_model_2.invoke(prompt)
    ans = f"Issue is {result.issue_type} , the tone of the customer is {result.tone} and the urgency level is {result.urgency}"
    return {'diagnosis': ans}

def negative_response(state:ReviewState):
    prompt = f"You are a support assistance. The user had a {state['diagnosis']} . Write an empathetic , helpful resolution message."
    result = model.invoke(prompt).content
    return {'response':result}

def positive_response(state:ReviewState):
    prompt = f"Write a warm thank you message in response to this review {state['review']}"
    result = model.invoke(prompt).content
    return {'response': result}

def check_condition(state:ReviewState)->Literal['diagonse_sentiment','positive_response']:
    if(state['sentiment']=='positive'):
        return positive_response
    else:
        return 'diagonse_sentiment'
    

In [ ]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('diagnose_sentiment',diagnose_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('negative_response',negative_response)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('check_condition',check_condition)
graph.add_edge('diagnose_sentiment','negative_response')

graph.add_edge('negative_response',END)
graph.add_edge('positive_response',END)

workflow = graph.compile()

In [ ]:
initial_state = {'review': "The product was excellent and exceeded my expectations."}
final_state = workflow(initial_state)
print(final_state)